In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!nvidia-smi

Wed Sep  2 12:38:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import subprocess
import sys

packages = [
    "vllm==0.6.*",
    "transformers==4.46.*",
    "accelerate==1.1.*",
    "httpx==0.27.*",
    "openai==1.54.*",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True
)

print("VLLM INSTALLATION COMPLETE")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 700.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.5/389.5 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
litellm 1.82.4 requires openai>=2.8.0, but you have openai 1.54.5 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
a2a-sdk 0.3.26 requires httpx>=0.28.1, but you have httpx 0.27.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is in

VLLM INSTALLATION COMPLETE


In [9]:
import os
import sys
import subprocess

SERVER_LOG = "/kaggle/working/server.log"

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"

log_file = open(SERVER_LOG, "w")

server = subprocess.Popen(
    [
        sys.executable,
        "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-1.5B-Instruct",
        "--dtype", "half",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
        "--port", "8000",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    env=environment,
)

print("SERVER LAUNCHED")
print("PID:", server.pid)
print("LOG:", SERVER_LOG)

SERVER LAUNCHED
PID: 190
LOG: /kaggle/working/server.log


In [10]:
import time
import urllib.request

print("Waiting for the model to load...")

for attempt in range(200):
    try:
        with urllib.request.urlopen(
            "http://localhost:8000/v1/models",
            timeout=5
        ) as response:
            if response.status == 200:
                print("SERVER HEALTHY: /v1/models -> 200")
                break
    except Exception:
        pass

    if server.poll() is not None:
        print("SERVER FAILED")
        with open("/kaggle/working/server.log", "r", errors="replace") as file:
            print(file.read()[-5000:])
        break

    time.sleep(3)
else:
    print("SERVER TIMEOUT")
    with open("/kaggle/working/server.log", "r", errors="replace") as file:
        print(file.read()[-5000:])

Waiting for the model to load...
SERVER FAILED
ria[rank0[rank0]:[W902 12:54:04.477953070 ProcessGroupNCCL.cpp:1250] Warning: WARNING: process group has NOT been destroyed before we destruct ProcessGroupNCCL. On normal program exit, the application should call destroy_process_group to ensure that any pending NCCL operations have finished in this process. In rare cases this process can exit before this point and block the progress of another member of the process group. This constraint has always been present,  but this warning has only been added since PyTorch 2.4 (function operatTask eTask exception was never retrieved
future: <Task finished name='Task-2' coro=<MQLLMEngineClient.run_output_handler_loop() done, defined at /usr/local/lib/python3.12/dist-packages/vllm/engine/multiprocessing/client.py:178> exception=ZMQError('Operation not supported')>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/vllm/engine/multiprocessing/clienTask exception was neve

In [11]:
import os
import sys
import subprocess

SERVER_LOG = "/kaggle/working/server_retry.log"

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"

log_file = open(SERVER_LOG, "w")

server = subprocess.Popen(
    [
        sys.executable,
        "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-1.5B-Instruct",
        "--dtype", "half",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
        "--disable-frontend-multiprocessing",
        "--port", "8000",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    env=environment,
)

print("SERVER RETRY LAUNCHED")
print("PID:", server.pid)
print("LOG:", SERVER_LOG)

SERVER RETRY LAUNCHED
PID: 819
LOG: /kaggle/working/server_retry.log


In [12]:
import time
import urllib.request

print("Waiting for the vLLM server...")

for attempt in range(200):
    try:
        with urllib.request.urlopen(
            "http://localhost:8000/v1/models",
            timeout=5
        ) as response:
            if response.status == 200:
                print("SERVER HEALTHY: /v1/models -> 200")
                break
    except Exception:
        pass

    if server.poll() is not None:
        print("SERVER FAILED")
        with open(
            "/kaggle/working/server_retry.log",
            "r",
            errors="replace"
        ) as file:
            print(file.read()[-5000:])
        break

    time.sleep(3)
else:
    print("SERVER TIMEOUT")
    with open(
        "/kaggle/working/server_retry.log",
        "r",
        errors="replace"
    ) as file:
        print(file.read()[-5000:])

Waiting for the vLLM server...
SERVER HEALTHY: /v1/models -> 200


In [13]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="sk-local"
)

response = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "In one sentence, what is a GPU?"
        }
    ],
    max_tokens=64,
    temperature=0.0
)

print(response.choices[0].message.content)
print("OPENAI-COMPATIBLE CLIENT: PASS")

A GPU, or Graphics Processing Unit, is a specialized processor designed to accelerate the performance of graphics and computational tasks in computers and other devices.
OPENAI-COMPATIBLE CLIENT: PASS


In [18]:
import asyncio
import time
import httpx

FIXED_PROMPTS = [
    "In one sentence, what is a GPU?",
    "List three reasons decode is memory-bound.",
    "Explain the KV cache to a new ops engineer in two sentences.",
    "What does continuous batching change versus static batching?",
    "Give a one-line definition of tokens per second.",
    "Why does a longer prompt increase time to first token?",
    "Name two things quantisation trades away for smaller memory.",
    "Summarise what an inference server does in three short bullets.",
]

QUEUE = [32, 32, 32, 256] * 6

async def one_request(client, prompt, max_tokens):
    payload = {
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": False,
    }

    response = await client.post(
        "http://localhost:8000/v1/chat/completions",
        json=payload
    )
    response.raise_for_status()

    body = response.json()
    return body["usage"]["completion_tokens"]

async def run_level(client, concurrency):
    semaphore = asyncio.Semaphore(concurrency)

    async def guarded(index):
        async with semaphore:
            return await one_request(
                client,
                FIXED_PROMPTS[index % len(FIXED_PROMPTS)],
                QUEUE[index]
            )

    start = time.time()

    counts = await asyncio.gather(
        *[guarded(index) for index in range(24)]
    )

    elapsed = time.time() - start

    return {
        "concurrency": concurrency,
        "requests": 24,
        "tokens_per_s": round(sum(counts) / elapsed, 1),
        "wall_s": round(elapsed, 3),
    }

async def run_sweep():
    results = []

    async with httpx.AsyncClient(timeout=180.0) as client:
        for index in range(4):
            await one_request(
                client,
                FIXED_PROMPTS[index],
                32
            )

        for concurrency in (1, 4, 8):
            result = await run_level(client, concurrency)
            print("LEVEL:", result)
            results.append(result)

    return results

In [19]:
vllm_results = await run_sweep()

print("\nVLLM SWEEP COMPLETE")

for result in vllm_results:
    print(result)

LEVEL: {'concurrency': 1, 'requests': 24, 'tokens_per_s': 33.9, 'wall_s': 40.961}
LEVEL: {'concurrency': 4, 'requests': 24, 'tokens_per_s': 154.1, 'wall_s': 9.013}
LEVEL: {'concurrency': 8, 'requests': 24, 'tokens_per_s': 228.9, 'wall_s': 6.069}

VLLM SWEEP COMPLETE
{'concurrency': 1, 'requests': 24, 'tokens_per_s': 33.9, 'wall_s': 40.961}
{'concurrency': 4, 'requests': 24, 'tokens_per_s': 154.1, 'wall_s': 9.013}
{'concurrency': 8, 'requests': 24, 'tokens_per_s': 228.9, 'wall_s': 6.069}


In [20]:
import json

baseline = {
    "1": 34.0,
    "4": 49.8,
    "8": 96.6
}

vllm = {
    "1": 33.9,
    "4": 154.1,
    "8": 228.9
}

speedup_by_concurrency = {
    level: round(vllm[level] / baseline[level], 2)
    for level in baseline
}

static_scaling_1_to_8 = round(baseline["8"] / baseline["1"], 2)
vllm_scaling_1_to_8 = round(vllm["8"] / vllm["1"], 2)

report = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "dtype": "fp16",
    "baseline_source": "Course sample baseline",
    "baseline": baseline,
    "vllm": vllm,
    "speedup_by_concurrency": speedup_by_concurrency,
    "static_scaling_1_to_8": static_scaling_1_to_8,
    "vllm_scaling_1_to_8": vllm_scaling_1_to_8,
    "prediction": "vLLM will achieve a higher 1-to-8 scaling multiple than static batching."
}

with open("/kaggle/working/ab_report.json", "w") as file:
    json.dump(report, file, indent=2)

print(json.dumps(report, indent=2))
print("AB REPORT CREATED")

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "dtype": "fp16",
  "baseline_source": "Course sample baseline",
  "baseline": {
    "1": 34.0,
    "4": 49.8,
    "8": 96.6
  },
  "vllm": {
    "1": 33.9,
    "4": 154.1,
    "8": 228.9
  },
  "speedup_by_concurrency": {
    "1": 1.0,
    "4": 3.09,
    "8": 2.37
  },
  "static_scaling_1_to_8": 2.84,
  "vllm_scaling_1_to_8": 6.75,
  "prediction": "vLLM will achieve a higher 1-to-8 scaling multiple than static batching."
}
AB REPORT CREATED


In [21]:
import json
import os

report_path = "/kaggle/working/ab_report.json"

if not os.path.exists(report_path):
    print("GREEN CHECK: FAIL (ab_report.json not found)")
else:
    with open(report_path, "r") as file:
        report = json.load(file)

    required_keys = {
        "baseline",
        "vllm",
        "speedup_by_concurrency"
    }

    missing_keys = required_keys - set(report)

    if missing_keys:
        print(
            f"GREEN CHECK: FAIL "
            f"(missing keys: {sorted(missing_keys)})"
        )
    else:
        baseline_8 = report["baseline"]["8"]
        vllm_8 = report["vllm"]["8"]
        speedup_8 = report["speedup_by_concurrency"]["8"]
        expected_speedup = vllm_8 / baseline_8

        if vllm_8 <= baseline_8:
            print(
                "GREEN CHECK: FAIL "
                "(vLLM concurrency-8 did not beat baseline)"
            )
        elif abs(speedup_8 - expected_speedup) > 0.1:
            print(
                "GREEN CHECK: FAIL "
                "(incorrect concurrency-8 speedup)"
            )
        else:
            print("baseline batch-8:", baseline_8)
            print("vLLM concurrency-8:", vllm_8)
            print("speedup at 8:", speedup_8, "x")
            print("GREEN CHECK: PASS")

baseline batch-8: 96.6
vLLM concurrency-8: 228.9
speedup at 8: 2.37 x
GREEN CHECK: PASS


In [22]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/ab_report.json"))

/kaggle/working/ab_report.json